# Gemini 3 Batch Transcription Pipeline

This Colab orchestrates the transcription of audio segments using Google's **Gemini 3** model.

**Runtime:** requires the repo on disk with the `radio-transcription-model` package installed from `model/` in editable mode (the `model/notebook_docker/` compose service does this on startup). The stock *Open in Colab* badge was removed; this notebook is no longer one-click Colab.

### Workflow Overview:
1.  **Job Submission**: Reads a JSONL manifest from GCS and submits asynchronous batch jobs in chunks of 15 segments. It automatically checks for existing transcripts and skips already processed files to avoid duplicate work and optimize API costs (Delta processing).
2.  **Organization**: Results are written directly to GCS.

In [ ]:
# @title Bootstrap and Install Environment
import os

# Clone repository if not already present (for hosted Colab environments)
if not os.path.exists("radio-transcription"):
    !git clone -q https://github.com/watch-duty/radio-transcription.git
    # Sync repository to the evaluation branch in Colab
    !cd radio-transcription && git fetch origin && git checkout experiment/context-window-evaluation && git pull

# Install the model library in editable mode along with required dependencies
try:
    import common

    print("✅ Library 'common' already installed.")
except ImportError:
    print("Installing library and dependencies...")
    %pip install -q -e radio-transcription/model[vertex] loguru tqdm

    import site
    import importlib

    importlib.reload(site)
    print("\n✅ Dependencies installed successfully.")

In [ ]:
import json
import re
import sys
import time

from google import genai
from google.cloud import storage
from google.colab import auth
from loguru import logger

from common.gemini.prompts import GEMINI_TRANSCRIBE_SYSTEM_PROMPT
from common.gemini.prompts import GEMINI_TRANSCRIBE_USER_PROMPT
from common.gemini.vertex import (
    GEMINI_GENERATION_CONFIG,
    GEMINI_SAFETY_SETTINGS,
)
from common.gemini.vertex import build_request, submit_batch_inference

In [ ]:
# @title Define constants and initial logging
from google.colab import userdata
import re
import sys

MODEL_ID = "gemini-3.1-flash-lite"  # @param ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {type:"string"}

GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET = userdata.get("GCS_BUCKET")

# @markdown ### 1. Input Audio Source
# @markdown Path or URI to the audio segmentation manifest (accepts full `gs://` URI or relative bucket path):
INPUT_AUDIO_MANIFEST_URI = ""  # @param {type:"string"}

# @markdown ### 2. Output Path Assembly (Choice 2: Auto-Constructed)
# @markdown Your final output directory is automatically assembled as:
# @markdown `gs://{GCS_BUCKET}/{TRANSCRIPTS_BASE_DIR}/{MODEL_ID}/{EXPERIMENT_NAME}/`
# @markdown *(⚠️ Do NOT include the model ID or experiment name in the base directory box below!)*
TRANSCRIPTS_BASE_DIR = ""  # @param {type:"string"}
EXPERIMENT_NAME = ""  # @param {type:"string"}

assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be in Colab secrets or provided."
assert GCS_BUCKET, "GCS_BUCKET must be in Colab secrets or provided."
assert INPUT_AUDIO_MANIFEST_URI, "INPUT_AUDIO_MANIFEST_URI must be provided."
assert TRANSCRIPTS_BASE_DIR, "TRANSCRIPTS_BASE_DIR must be provided."
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided."


# Universal Path Normalization
def _normalize_gcs_uri(path_or_uri: str, bucket_name: str) -> str:
    path_or_uri = path_or_uri.strip()
    if path_or_uri.startswith("gs://"):
        return path_or_uri
    clean_path = path_or_uri.lstrip("/")
    return f"gs://{bucket_name}/{clean_path}"


MANIFEST_URI = _normalize_gcs_uri(INPUT_AUDIO_MANIFEST_URI, GCS_BUCKET)

# Clean prefix (strip gs://bucket/ if user accidentally pasted a full URI, and strip leading/trailing slashes)
_clean_prefix = TRANSCRIPTS_BASE_DIR.strip()
if _clean_prefix.startswith("gs://"):
    _clean_prefix = _clean_prefix.replace(f"gs://{GCS_BUCKET}/", "")
_clean_prefix = _clean_prefix.strip("/")

# Create a model-specific directory name
MODEL_ID_DIR = re.sub(r"[-\.]", "_", MODEL_ID)
GCS_OUTPUT_BASE = f"{_clean_prefix}/{MODEL_ID_DIR}/{EXPERIMENT_NAME}"
GCP_LOCATION = "global"

BATCH_INPUT_URI = (
    f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/vertex_batch_input.jsonl"
)
BATCH_OUTPUT_ROOT = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/batch_results/"

# The consistent final path for consolidated results
CONSISTENT_OUTPUT_URI = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/predictions.jsonl"

# Define GCS_INPUT_DIR for metadata/retry logic (directory of the input manifest)
_manifest_parts = MANIFEST_URI.replace("gs://", "").split("/")
GCS_INPUT_DIR = "/".join(_manifest_parts[1:-1])

logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}")
print("=" * 60)
print(f"✅ Input Manifest URI:  {MANIFEST_URI}")
print(f"✅ GCS Input Dir:       {GCS_INPUT_DIR}")
print(f"✅ Output Predictions:  {CONSISTENT_OUTPUT_URI}")
print("=" * 60)

In [ ]:
# @title Allow access to the GCS bucket to the VertexAI Service Agent (uncomment code to run)
# !gcloud storage buckets add-iam-policy-binding gs://{GCS_BUCKET} \
#     --member="serviceAccount:service-$(gcloud projects describe {GCP_PROJECT_ID} --format='value(projectNumber)')@gcp-sa-aiplatform.iam.gserviceaccount.com" \
#     --role="roles/storage.objectViewer"

In [ ]:
# @title Authenticate with GCP
auth.authenticate_user()

!gcloud config set project {GCP_PROJECT_ID} --quiet

In [ ]:
# @title Experiment here — override canonical defaults for A/B testing
# Edit these to try prompt / inference-config variants in this session.
# Promote a winner by editing model/src/common/gemini/prompts.py or model/src/common/gemini/vertex.py and committing.

# SYSTEM_PROMPT = (
#     GEMINI_TRANSCRIBE_SYSTEM_PROMPT  # <- edit to A/B a prompt version
# )
# USER_PROMPT = GEMINI_TRANSCRIBE_USER_PROMPT
# GENERATION_CONFIG = {
#     **GEMINI_GENERATION_CONFIG
# }  # <- edit temperature / max_output_tokens
# SAFETY_SETTINGS = GEMINI_SAFETY_SETTINGS

SYSTEM_PROMPT = """\
Your primary task is to produce a strict, verbatim transcription of the spoken audio. Your absolute highest priority is to transcribe only what you hear with high acoustic certainty. Do not add, invent, or infer any speech that is not clearly audible. The audio may originate from VHF/UHF radio traffic and can include mic clicks, RF static, radio hum, and potentially unintelligible speech. When the audio is unequivocally confirmed as fire-related dispatch, speakers often use heavy jargon, and specific formatting rules apply.

CRITICAL RULES:
1. Output the transcript strictly and precisely as spoken in the audio, with no newlines. Do not add, invent, or infer any speech that is not clearly audible.
2. When transcribing numbers, write the digits grouped together (e.g., 100, 6333).
3. If the audio contains a unit identifier, format it as the unit type followed by digits (e.g., Engine 41, Battalion 2). Apply this rule strictly only if the unit identifier is clearly spoken AND the context is unequivocally fire-related dispatch.
4. Transcribe only the duration of speech present. Do not extend the transcription with additional words or phrases that were not spoken, even if contextually plausible.

QUALITY GATE: Your absolute highest priority is to transcribe only what you hear with high acoustic certainty.
    *   If the audio contains clear speech that is not fire-related dispatch, you MUST transcribe it verbatim, exactly as heard, without applying any fire-specific formatting or jargon, and without attempting to interpret it as fire dispatch traffic.
    *   If a portion of audio is obscured, noisy, ambiguous, or contains speech that cannot be confidently identified, you MUST replace that specific portion with [UNINTELLIGIBLE].
    *   Do not attempt to infer, guess, or invent speech to fit any expected context or terminology list.
    *   Do not attempt to phonetically guess ambiguous noise.
    *   If the entire audio segment does not contain any discernible speech, output only [UNINTELLIGIBLE].

TASK:
Transcribe the audio file verbatim.
"""

SAFETY_SETTINGS = [
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_CIVIC_INTEGRITY", "threshold": "BLOCK_NONE"},
]

GENERATION_CONFIG = {
    "temperature": 0.0,
    "max_output_tokens": 512,
    "thinking_config": {"thinking_budget": 0},
}

USER_PROMPT = "Transcribe the provided audio verbatim according to the system instructions."

In [ ]:
# @title Helper functions
def get_failed_uris(results_uri: str, bucket_name: str) -> list[str]:
    """Identifies URIs that resulted in errors in the final output file."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    path = results_uri.replace(f"gs://{bucket_name}/", "")
    blob = storage_client.bucket(bucket_name).blob(path)
    if not blob.exists():
        return []
    lines = blob.download_as_text().strip().split("\n")
    failed = []
    for line in lines:
        if not line.strip():
            continue
        data = json.loads(line)
        if data.get("status"):
            parts = data["request"]["contents"][0]["parts"]
            uri = next(
                (
                    p.get("file_data", {}).get("file_uri")
                    or p.get("fileData", {}).get("fileUri")
                    for p in parts
                    if "file_uri" in str(p) or "fileUri" in str(p)
                ),
                "unknown",
            )
            failed.append(uri)
    return failed


def create_retry_manifest(
    failed_uris: list[str], original_manifest_uri: str, retry_manifest_uri: str
) -> None:
    """Filters original manifest and wraps in the correct 'request' structure for Vertex Batch."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    bucket_name = original_manifest_uri.replace("gs://", "").split("/")[0]
    path = "/".join(original_manifest_uri.replace("gs://", "").split("/")[1:])
    content = (
        storage_client.bucket(bucket_name)
        .blob(path)
        .download_as_text()
        .strip()
        .split("\n")
    )

    retry_entries = []
    for line in content:
        if not line.strip():
            continue
        entry = json.loads(line)
        if entry["audio_filepath"] in failed_uris:
            batch_entry = build_request(
                entry["audio_filepath"],
                system_prompt=SYSTEM_PROMPT,
                user_prompt=USER_PROMPT,
                generation_config=GENERATION_CONFIG,
                safety_settings=SAFETY_SETTINGS,
            )
            retry_entries.append(json.dumps(batch_entry))

    if retry_entries:
        out_bucket = retry_manifest_uri.replace("gs://", "").split("/")[0]
        out_path = "/".join(
            retry_manifest_uri.replace("gs://", "").split("/")[1:]
        )
        storage_client.bucket(out_bucket).blob(out_path).upload_from_string(
            "\n".join(retry_entries)
        )
        logger.info(
            f"Uploaded retry manifest with {len(retry_entries)} entries."
        )


def prepare_batch_manifest(
    input_manifest_uri: str,
    output_batch_manifest_uri: str,
    *,
    overwrite: bool = False,
    limit: int | None = None,
) -> str | None:
    """Prepares the JSONL manifest with correct structural requirements for Vertex Batch."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    m_bucket = input_manifest_uri.replace("gs://", "").split("/")[0]
    m_path = "/".join(input_manifest_uri.replace("gs://", "").split("/")[1:])

    try:
        manifest_blob = storage_client.bucket(m_bucket).blob(m_path)
        if not manifest_blob.exists():
            logger.error(
                f"Manifest not found at {input_manifest_uri}. Check AUDIO_PREPROCESSING settings."
            )
            return None
        manifest_content = manifest_blob.download_as_text().strip().split("\n")
    except Exception as e:
        logger.error(f"Error reading manifest: {e}")
        return None

    processed_uris = set()
    if not overwrite:
        r_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
        r_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        results_blob = storage_client.bucket(r_bucket).blob(r_path)
        if results_blob.exists():
            for line in results_blob.download_as_text().strip().split("\n"):
                if line.strip():
                    data = json.loads(line)
                    if not data.get("status"):
                        contents = data["request"].get("contents", [])
                        parts = (
                            contents[0].get("parts", [])
                            if isinstance(contents, list) and contents
                            else contents.get("parts", [])
                        )
                        uri = next(
                            (
                                p.get("file_data", {}).get("file_uri")
                                or p.get("fileData", {}).get("fileUri")
                                for p in parts
                                if "file_uri" in str(p) or "fileUri" in str(p)
                            ),
                            None,
                        )
                        if uri:
                            processed_uris.add(uri)

    batch_entries = []
    for line in manifest_content:
        if not line.strip():
            continue
        entry = json.loads(line)
        if entry["audio_filepath"] in processed_uris:
            continue

        batch_entry = build_request(
            entry["audio_filepath"],
            system_prompt=SYSTEM_PROMPT,
            user_prompt=USER_PROMPT,
            generation_config=GENERATION_CONFIG,
            safety_settings=SAFETY_SETTINGS,
        )
        batch_entries.append(json.dumps(batch_entry))
        if limit and len(batch_entries) >= limit:
            logger.info(f"Test limit of {limit} reached.")
            break

    if not batch_entries:
        return None

    out_bucket = output_batch_manifest_uri.replace("gs://", "").split("/")[0]
    out_path = "/".join(
        output_batch_manifest_uri.replace("gs://", "").split("/")[1:]
    )
    storage_client.bucket(out_bucket).blob(out_path).upload_from_string(
        "\n".join(batch_entries)
    )
    logger.info(
        f"Prepared {len(batch_entries)} segments for processing at {output_batch_manifest_uri}"
    )
    return output_batch_manifest_uri


def run_automated_retry_pipeline() -> None:
    """Orchestrates checking for failures, creating a retry manifest, and merging results."""
    logger.info("Starting automated error check...")
    failed_uris = get_failed_uris(CONSISTENT_OUTPUT_URI, GCS_BUCKET)
    if not failed_uris:
        logger.info("No failed segments detected. Pipeline complete.")
        return

    logger.info(
        f"Detected {len(failed_uris)} failures. Creating retry manifest... "
    )
    RETRY_MANIFEST = (
        f"gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/automated_retry_manifest.jsonl"
    )
    create_retry_manifest(failed_uris, MANIFEST_URI, RETRY_MANIFEST)

    try:
        submit_batch_inference(
            input_uri=RETRY_MANIFEST,
            output_uri=BATCH_OUTPUT_ROOT,
            model=MODEL_ID,
            project=GCP_PROJECT_ID,
            location=GCP_LOCATION,
        )
        consolidate_all_successes(
            GCS_BUCKET, GCS_OUTPUT_BASE, CONSISTENT_OUTPUT_URI
        )
    except RuntimeError as e:
        logger.error(f"Retry job failed: {e}")
    validate_transcription_results(
        MANIFEST_URI, CONSISTENT_OUTPUT_URI, GCS_BUCKET
    )


def consolidate_all_successes(
    bucket_name: str, output_base: str, target_uri: str
) -> None:
    """Scans all batch result folders and builds a unique map of successful transcriptions."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    bucket = storage_client.bucket(bucket_name)
    prefix = f"{output_base}/batch_results/"
    blobs = bucket.list_blobs(prefix=prefix)

    success_map = {}
    for blob in blobs:
        if "predictions.jsonl" not in blob.name:
            continue

        lines = blob.download_as_text().strip().split("\n")
        for line in lines:
            if not line.strip():
                continue
            data = json.loads(line)
            if not data.get("status"):
                parts = data["request"]["contents"][0]["parts"]
                uri = next(
                    (
                        p.get("file_data", {}).get("file_uri")
                        or p.get("fileData", {}).get("fileUri")
                        for p in parts
                        if "file_uri" in str(p) or "fileUri" in str(p)
                    ),
                    "unknown",
                )
                if uri not in success_map:
                    success_map[uri] = line

    target_path = target_uri.replace(f"gs://{bucket_name}/", "")
    bucket.blob(target_path).upload_from_string("\n".join(success_map.values()))
    logger.info(
        f"Consolidation complete. Total unique successful segments: {len(success_map)}"
    )


def validate_transcription_results(
    manifest_uri: str,
    results_uri: str,
    bucket_name: str,
    expected_count: int | None = None,
) -> None:
    storage_client = storage.Client(project=GCP_PROJECT_ID)

    if expected_count is None:
        m_bucket = manifest_uri.replace("gs://", "").split("/")[0]
        m_path = "/".join(manifest_uri.replace("gs://", "").split("/")[1:])

        m_blob = storage_client.bucket(m_bucket).blob(m_path)
        if not m_blob.exists():
            logger.warning(
                f"Validation skipped: Manifest {manifest_uri} does not exist."
            )
            return

        expected_count = len(
            [
                line
                for line in m_blob.download_as_text().strip().split("\n")
                if line.strip()
            ]
        )

    r_path = results_uri.replace(f"gs://{bucket_name}/", "")
    results_blob = storage_client.bucket(bucket_name).blob(r_path)
    if results_blob.exists():
        found = len(
            [
                line
                for line in results_blob.download_as_text().strip().split("\n")
                if line.strip()
            ]
        )
        if found == expected_count:
            logger.info(
                f"Pipeline Validation: SUCCESS. Expected {expected_count}, Found {found}."
            )
        else:
            logger.error(
                f"Pipeline Validation: FAILURE. Expected {expected_count}, Found {found}."
            )

In [ ]:
# @title Main Job Submission
TEST_RUN = False  # @param {type:"boolean"}
TEST_LIMIT = 2  # @param {type:"integer"}

actual_batch_input = prepare_batch_manifest(
    input_manifest_uri=MANIFEST_URI,
    output_batch_manifest_uri=BATCH_INPUT_URI,
    overwrite=OVERWRITE_EXISTING,
    limit=TEST_LIMIT if TEST_RUN else None,
)

if actual_batch_input:
    logger.info("Submitting main batch job...")
    try:
        submit_batch_inference(
            input_uri=actual_batch_input,
            output_uri=BATCH_OUTPUT_ROOT,
            model=MODEL_ID,
            project=GCP_PROJECT_ID,
            location=GCP_LOCATION,
        )
        logger.info("Main job completed. Consolidating results...")
        consolidate_all_successes(
            GCS_BUCKET, GCS_OUTPUT_BASE, CONSISTENT_OUTPUT_URI
        )
        validate_transcription_results(
            MANIFEST_URI, CONSISTENT_OUTPUT_URI, GCS_BUCKET
        )
    except RuntimeError as e:
        logger.error(f"Main job failed: {e}. Skipping consolidation.")
else:
    logger.info(
        "Skipping main job submission (no new segments to process or manifest missing). Attempting consolidation of previous results..."
    )
    consolidate_all_successes(
        GCS_BUCKET, GCS_OUTPUT_BASE, CONSISTENT_OUTPUT_URI
    )
    validate_transcription_results(
        MANIFEST_URI, CONSISTENT_OUTPUT_URI, GCS_BUCKET
    )

In [ ]:
# @title Automated Retry & Merge Orchestrator
logger.info("Executing automated retry and merge pipeline...")
run_automated_retry_pipeline()

In [ ]:
# @title Write run metadata
import datetime as _datetime

_run_metadata = {
    "prompt": SYSTEM_PROMPT,
    "model_id": MODEL_ID,
    # "audio_preprocessing": AUDIO_PREPROCESSING,
    "generation_config": GENERATION_CONFIG,
    "safety_settings": SAFETY_SETTINGS,
    "project_name": TRANSCRIPTS_BASE_DIR.split("/")[0]
    if TRANSCRIPTS_BASE_DIR
    else "unknown",
    "experiment_name": EXPERIMENT_NAME,
    "input_manifest_uri": MANIFEST_URI,
    "predictions_uri": CONSISTENT_OUTPUT_URI,
    "run_timestamp_utc": _datetime.datetime.now(
        _datetime.timezone.utc
    ).strftime("%Y-%m-%dT%H:%M:%S.%fZ"),
}

_meta_path = f"{GCS_OUTPUT_BASE}/run_metadata.json"
storage.Client(project=GCP_PROJECT_ID).bucket(GCS_BUCKET).blob(
    _meta_path
).upload_from_string(
    json.dumps(_run_metadata, indent=2),
    content_type="application/json",
)
logger.info(f"Run metadata written to gs://{GCS_BUCKET}/{_meta_path}")